In [ ]:
import json
from functools import lru_cache
from pathlib import Path
from typing import Any

from langchain.chat_models import init_chat_model
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel

from core.settings import settings
from workflow.builder import format_messages, get_last_human_message
from workflow.state import AgentState

# intent/intent.md 不含 service_type，由商品匹配结果决定
class IntentResult(BaseModel):
    intent: str
    room_number: str | None = None
    product: str | None = None
    fault: str | None = None
    area: str | None = None
    urgency: str | None = None
    expected_start_time: str | None = None
    goods_arrival_status: str | None = None
    managed_repair_scope: str | None = None
    user_confirmed: bool = False
    user_cancelled: bool = False

try:
    PROMPTS_DIR = Path(__file__).resolve().parents[1] / "prompts"
except NameError:
    PROMPTS_DIR = Path.cwd() / "prompts"

print("PROMPTS_DIR:", PROMPTS_DIR)

def to_prompt_text(value: object) -> str:
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, default=str)

def load_prompt(relative_path: str) -> str:
    return (PROMPTS_DIR / relative_path).read_text(encoding="utf-8")

def render_prompt(relative_path: str, **variables: object) -> str:
    prompt = load_prompt(relative_path)
    for key, value in variables.items():
        prompt = prompt.replace(f"{{{{{key}}}}}", to_prompt_text(value))
    return prompt

@lru_cache
def get_llm() -> BaseChatModel:
    return init_chat_model(
        model=settings.openai_model,
        model_provider="openai",
        base_url=settings.openai_base_url,
        api_key=settings.openai_api_key,
        temperature=settings.openai_temperature,
        model_kwargs={"extra_body": {"enable_thinking": False}},
    )

async def intent_node(state: AgentState) -> Any:
    llm = get_llm().with_structured_output(IntentResult)
    result = await llm.ainvoke([
        SystemMessage(content=render_prompt(
            "intent/intent.md",
            conversation_history=format_messages(state["messages"]),
            user_input=get_last_human_message(state["messages"]),
            status=state.get("status") or "idle",
            last_order=state.get("last_order", {}),
        ))
    ])
    return result


In [ ]:
import asyncio

# 五类订单的典型用户输入
test_cases = [
    # (描述, 期望 intent, 期望抽取到的字段)
    ("1208房空调不制冷，比较急",        "create_order", "room_number=1208, product=空调, fault=不制冷"),
    ("大堂走廊灯不亮",                  "create_order", "managed_repair_scope=公区"),
    ("帮我安装洗衣机，明天上午，货到了", "create_order", "product=洗衣机, expected_start_time=明天上午, goods_arrival_status=已到场"),
    ("量一下 306 房窗帘尺寸，本周五",   "create_order", "product=窗帘, expected_start_time=本周五"),
    ("我要点餐",                        "smalltalk",    "—"),
    ("取消",                            "cancel_order", "user_cancelled=True"),
]

for user_input, expected_intent, note in test_cases:
    print(f"\n{'='*60}")
    print(f"输入：{user_input}")
    print(f"期望 intent：{expected_intent}  |  关注字段：{note}")
    state: AgentState = {
        "messages": [HumanMessage(content=user_input)],
        "last_user_message": user_input,
    }
    result = await intent_node(state)
    print(f"结果：{result}")
